In [ ]:
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import part1 # to use code from tutorial

In [ ]:
def get_vocab(texts, min_count=5, max_size=None):
    counts = Counter(item for text in texts for item in text.split())
    kept = [item for item, count in counts.most_common(max_size) if count >= min_count]
    return {item: i for i, item in enumerate(kept)}

In [ ]:
# template for encoder classes
class Encoder:
    def __len__(self) -> int:
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

In [ ]:
class TokenIndicator(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        for item in set(x.split()):
            index = self.vocab.get(item)
            if index is not None:
                vector[index] = 1
        return vector

In [ ]:
# to align with tutorial
def make_pairs(df, column, label_names, label_column="Party_orientation"):
    label_to_id = {name: i for i, name in enumerate(label_names)}
    texts = df[column].str.join(" ")
    labels = [label_to_id[name] for name in df[label_column]]
    return list(zip(texts, labels))

Testing encoder

In [ ]:
# get the dataframe
df = pd.read_pickle("./sampled_dataframe.pkl")

# get the label names
label_names = sorted(df["Party_orientation"].unique())
print(len(label_names), label_names)

In [ ]:
# testing the functions
column = "tokens"   # "tokens" or "lemmas"

pairs = make_pairs(df, column, label_names)
vocab = get_vocab([text for text, _ in pairs], min_count=5)
encoder = TokenIndicator(vocab)

print("vocabulary size:", len(encoder))

In [ ]:
# an example
text, label = pairs[2]
vector = encoder(text)
inverse_vocab = {i: item for item, i in vocab.items()}

print(text)
print("label:", label, label_names[label])
print("active items:", [inverse_vocab[i] for i in vector.nonzero()[0]])

In [ ]:
# extra
# effect of min_count on the vocabulary size
all_texts = [text for text, _ in pairs]
for m in [1, 5, 20, 100]:
    print(m, len(get_vocab(all_texts, min_count=m)))

Model architecture

In [ ]:
results_folder = Path("./results")
results_folder.mkdir(exist_ok=True)

my_seeds = [0]
num_folds = 5

In [ ]:
# load data and take just a small sample for testing
df = pd.read_pickle("./sampled_dataframe.pkl")

label_names = sorted(df["Party_orientation"].unique())
outcome_space = part1.clslib.NamedClasses(label_names)
print(len(outcome_space), label_names)

small = df.sample(10000, random_state=0)
print(small["Party_orientation"].value_counts())

In [ ]:
# get vocab and encoding
column = "lemmas"

pairs = make_pairs(small, column, label_names)
vocab = get_vocab([text for text, _ in pairs], min_count=5, max_size=5000)
encoder = TokenIndicator(vocab)

print("vocabulary size:", len(encoder))

In [ ]:
# like in tutorial
my_cross_validation_sets = []
for seed in my_seeds:
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(pairs=pairs, k=num_folds, seed=seed)
    )

fold0 = my_cross_validation_sets[0][0]
print(len(fold0["training"]), "training ,", len(fold0["validation"]), "validation")

Crossvalidation

In [ ]:
my_classifiers = {}
my_classifiers["mlp_indicator-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_indicator-lemmas/seed-{seed}"
    my_classifiers["mlp_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

In [ ]:
accuracies = my_classifiers["mlp_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

Single fold training for later tuning maybe

In [ ]:
%%time

metrics_for_all, best_epoch, best_preds = part1.smlp.train_neural_network_model(
    fold0["training"],
    fold0["validation"],
    num_classes=len(outcome_space),
    encoder=encoder,
    save_to=f"{results_folder}/tmpmodel",
    hidden_dim=128,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-4,
    weight_decay=1e-4,
    num_epochs=30,
    device="cpu",
)

# checking values
best = metrics_for_all[best_epoch]
print("Best epoch:", best_epoch)
print("Train:", best["train_metrics"])
print("Validation:", best["val_metrics"])